# **Indian Startup Ecosystem Analytics**
*(Data Cleaning, Standardization & SQL-Based Business Analysis)*  

### About the Dataset

**Source:** Kaggle  
**Dataset:** [Indian Startups - Funding Data](https://www.kaggle.com/datasets/omkargowda/indian-startups-funding-data)

### Objective

The goal of this project is to clean, standardize, and analyze startup funding data collected between 2018 and 2021. Since the data spans multiple years with inconsistent schemas and formatting, the first objective is to prepare a unified dataset suitable for analysis.

The project aims to answer questions such as:

* How has startup funding changed over the years?
* Which sectors received the highest investment?
* Which cities have the largest startup ecosystem?
* Which startups raised the most funding?
* How does funding vary across different investment stages?

The project includes data cleaning and preprocessing using Python (Pandas), followed by SQL-based exploratory analysis to uncover funding trends and business insights.

In [1]:
#importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os

In [2]:
# Extracting the zip file
with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('startup_data')

In [3]:
#checking the files inside the zip file
print(os.listdir('startup_data'))

['startup_funding2020.csv', 'startup_funding2019.csv', 'startup_funding2021.csv', 'startup_funding2018.csv']


In [4]:
#reading the csv files
df18 = pd.read_csv('startup_data/startup_funding2018.csv')
df19 = pd.read_csv('startup_data/startup_funding2019.csv')
df20 = pd.read_csv('startup_data/startup_funding2020.csv')
df21 = pd.read_csv('startup_data/startup_funding2021.csv')

In [5]:
df18.columns

Index(['Company Name', 'Industry', 'Round/Series', 'Amount', 'Location',
       'About Company'],
      dtype='object')

In [6]:
df18.columns=df18.columns.str.lower().str.replace(' ','_').str.replace('/','_')
df19.columns=df19.columns.str.lower().str.replace(' ','_').str.replace('/','_')
df20.columns=df20.columns.str.lower().str.replace(' ','_').str.replace('/','_')
df21.columns=df21.columns.str.lower().str.replace(' ','_').str.replace('/','_')

In [7]:
#adding a year column before combining the 4 csv files for better clarity
dfs=[df18,df19,df20,df21]
years=[2018,2019,2020,2021]
for df,year in zip(dfs,years):
  df['year']=year

In [8]:
#checking the dimensions of the 4 files before combining
for year, df in zip([2018,2019,2020,2021], [df18,df19,df20,df21]):
    print(f"{year}: {df.shape}")

2018: (526, 7)
2019: (89, 10)
2020: (1055, 11)
2021: (1209, 10)


In [9]:
#checking for column names
pd.DataFrame({
    '2018': pd.Series(df18.columns),
    '2019': pd.Series(df19.columns),
    '2020': pd.Series(df20.columns),
    '2021': pd.Series(df21.columns)
})

,2018,2019,2020,2021
0,company_name,company_brand,company_brand,company_brand
1,industry,founded,founded,founded
2,round_series,headquarter,headquarter,headquarter
3,amount,sector,sector,sector
4,location,what_it_does,what_it_does,what_it_does
5,about_company,founders,founders,founders
6,year,investor,investor,investor
7,NaN,amount($),amount($),amount($)
8,NaN,stage,stage,stage
9,NaN,year,unnamed:_9,year


In [10]:
#comparing if differently names columns represent the same information
print('location vs headquarter')
print(pd.DataFrame({
    '2018':df18['location'],
    '2019':df19['headquarter'],
    '2020':df20['headquarter'],
    '2021':df21['headquarter']
    }).head())

print('\nindustry vs sector')
print(pd.DataFrame({
    '2018':df18['industry'],
    '2019':df19['sector'],
    '2020':df20['sector'],
    '2021':df21['sector']
    }).head())

print('\nround series vs stage')
print(pd.DataFrame({
    '2018':df18['round_series'],
    '2019':df19['stage'],
    '2020':df20['stage'],
    '2021':df21['stage']
    }).head())

print('\namount vs amount($)')
print(pd.DataFrame({
    '2018':df18['amount'],
    '2019':df19['amount($)'],
    '2020':df20['amount($)'],
    '2021':df21['amount($)']
    }).head())

location vs headquarter
                               2018       2019       2020       2021
0       Bangalore, Karnataka, India        NaN    Chennai  Bangalore
1        Mumbai, Maharashtra, India     Mumbai  Bangalore     Mumbai
2           Gurgaon, Haryana, India     Mumbai       Pune     Mumbai
3       Noida, Uttar Pradesh, India    Chennai  New Delhi     Mumbai
4  Hyderabad, Andhra Pradesh, India  Telangana     Indore   Gurugram

industry vs sector
                                                2018             2019  \
0  Brand Marketing, Event Promotion, Marketing, S...        Ecommerce   
1                               Agriculture, Farming           Edtech   
2   Credit, Financial Services, Lending, Marketplace           Edtech   
3                        Financial Services, FinTech  Interior design   
4                 E-Commerce Platforms, Retail, SaaS         AgriTech   

                 2020            2021  
0            AgriTech      AI startup  
1              EdTech  

In [11]:
#investigating the extra column in df20
df20['unnamed:_9'].isnull().sum()

np.int64(1053)

In [12]:
df20[df20['unnamed:_9'].notna()]

,company_brand,founded,headquarter,sector,what_it_does,founders,investor,amount($),stage,unnamed:_9,year
611,Walrus,2019,Bangalore,Fintech,It provides banking solutions for teens and yo...,"Bhagaban Behera, Sriharsha Shetty, Nakul Kelkar",Better Capital,Undisclosed,Pre-Seed,Pre-Seed,2020
613,goDutch,NaN,Mumbai,Fintech,Group Payments platform,"Aniruddh Singh, Riyaz Khan, Sagar Sheth","Matrix India, Y Combinator, Global Founders Ca...","$1,700,000",Seed Round,Seed Round,2020


In [13]:
df20.drop(columns=['unnamed:_9'], inplace=True)

### ***Observations***
* The differences in number of columns is due to the  schema inconsistencies rather than entirely new features.
* Similar attributes are represented using different column names across years.\
eg:\
company_name → company_brand\
industry → sector\
round_series → stage\
amount → amount($)\
location → headquarter\
about_company → what_it_does
* The datasets for 2019–2021 contain additional columns (founded, founders, and investor) that are absent in the 2018 dataset.
* The dataset for 2020 had an additional column 'unnamed:_9' which is mostly null values. On further inspection of the two non-null entries, it is found that they are the duplicates of the 'stage' value and hence is dropped




In [14]:
#renaming columns before combining
df18.rename(columns={'company_name':'company_brand', 'industry':'sector', 'round_series':'stage', 'amount':'amount($)', 'location':'headquarter',
       'about_company':'what_it_does'},inplace=True)

In [15]:
#combining the files and extracting only common attributes
df=pd.concat([df18,df19,df20,df21],ignore_index=True)
df=df[['company_brand','sector','stage','amount($)','headquarter','what_it_does','year']]

In [16]:
df.head()

,company_brand,sector,stage,amount($),headquarter,what_it_does,year
0,TheCollegeFever,"Brand Marketing, Event Promotion, Marketing, S...",Seed,250000,"Bangalore, Karnataka, India","TheCollegeFever is a hub for fun, fiesta and f...",2018
1,Happy Cow Dairy,"Agriculture, Farming",Seed,"₹40,000,000","Mumbai, Maharashtra, India",A startup which aggregates milk from dairy far...,2018
2,MyLoanCare,"Credit, Financial Services, Lending, Marketplace",Series A,"₹65,000,000","Gurgaon, Haryana, India",Leading Online Loans Marketplace in India,2018
3,PayMe India,"Financial Services, FinTech",Angel,2000000,"Noida, Uttar Pradesh, India",PayMe India is an innovative FinTech organizat...,2018
4,Eunimart,"E-Commerce Platforms, Retail, SaaS",Seed,—,"Hyderabad, Andhra Pradesh, India",Eunimart is a one stop solution for merchants ...,2018


In [17]:
df.shape

(2879, 7)

In [18]:
df.columns

Index(['company_brand', 'sector', 'stage', 'amount($)', 'headquarter',
       'what_it_does', 'year'],
      dtype='object')

## **Standardizing Company brands**

In [19]:

df['company_brand']=df['company_brand'].astype(str).str.strip().str.lower().str.replace(r'[^a-z0-9 ]','',regex=True).str.replace(r'\s+', ' ', regex=True)

In [20]:
df['company_brand'].value_counts().head(50)

,count
company_brand,
byjus,12
bharatpe,10
nykaa,7
zomato,7
mpl,6
oyo,6
zetwerk,6
spinny,6
vedantu,6


In [21]:
df[df['company_brand'].str.contains('byju', na=False)]

,company_brand,sector,stage,amount($),headquarter,what_it_does,year
130,byjus,"EdTech, Education, Higher Education, Secondary...",Private Equity,"$540,000,000","Bangalore, Karnataka, India",BYJU’s is an edtech company that is reinventin...,2018
542,byjus,Edtech,NaN,"$540,000,000",NaN,Provides online learning classes,2019
739,byjus,EdTech,NaN,"$200,000,000",Bangalore,BYJU'S is an educational technology company th...,2020
941,byjus,EdTech,NaN,"$500,000,000",Bangalore,An Indian educational technology and online tu...,2020
977,byjus,EdTech,NaN,"$500,000,000",Bangalore,An Indian educational technology and online tu...,2020
1109,byjus,EdTech,NaN,"$122,000,000",Bangalore,Provides online learning classes,2020
1233,byjus,Edtech,NaN,Undisclosed,Bangalore,Provides online learning classes,2020
1550,byju,Edtech,NaN,"$200,000,000",NaN,Provides online learning classes,2020
1650,byju,Edtech,NaN,"$200,000,000",NaN,Provides online learning classes,2020
2308,byjus,EdTech,NaN,$350000000,Bangalore,BYJU'S is an educational technology company th...,2021


In [22]:
df['company_brand']=df['company_brand'].replace({'byju':'byjus'})

In [23]:
df['company_brand'].value_counts().head(50)

,count
company_brand,
byjus,14
bharatpe,10
zomato,7
nykaa,7
spinny,6
zetwerk,6
mpl,6
vedantu,6
trell,6


### ***Insight***
* Company names contained inconsistencies due to capitalization and special characters used
* Hence, they were standardized by by converting them to lower case and removing those special characters
* A sanity check was done using byjus to check if the different variations are handled

## **Standardizing Sectors**

In [24]:
df['sector']=df['sector'].astype(str).str.split(',').str[0]
df['sector'].value_counts().head(25)


,count
sector,
FinTech,175
EdTech,148
Financial Services,88
Fintech,85
Edtech,74
E-commerce,73
Automotive,54
AgriTech,43
Food & Beverages,39


In [25]:
#Standardising the top common sectors
sector_map = {
    'FinTech': 'Fintech',
    'EdTech': 'Edtech',
    'E-learning': 'Edtech',
    'HealthCare': 'Healthcare',
    'Health Care': 'Healthcare',
    'HealthTech': 'Healthtech',
    'SaaS startup': 'SaaS',
    'Saas':'SaaS',
    'Ai Startup':'AI Startup',
    '—': np.nan #replacing - with nulls
}

df['sector'] = df['sector'].str.strip().replace(sector_map)

In [26]:
df['stage'].unique()

array(['Seed', 'Series A', 'Angel', 'Series B', 'Pre-Seed',
       'Private Equity', 'Venture - Series Unknown', 'Grant',
       'Debt Financing', 'Post-IPO Debt', 'Series H', 'Series C',
       'Series E', 'Corporate Round', 'Undisclosed',
       'https://docs.google.com/spreadsheets/d/1x9ziNeaz6auNChIHnMI8U6kS7knTr3byy_YBGfQaoUA/edit#gid=1861303593',
       'Series D', 'Secondary Market', 'Post-IPO Equity',
       'Non-equity Assistance', 'Funding Round', nan, 'Fresh funding',
       'Pre series A', 'Series G', 'Post series A', 'Seed funding',
       'Seed fund', 'Series F', 'Series B+', 'Seed round', 'Pre-series A',
       'Pre-seed', 'Pre-series', 'Debt', 'Pre-series C', 'Pre-series B',
       'Bridge', 'Series B2', 'Pre- series A', 'Edge', 'Pre-Series B',
       'Seed A', 'Series A-1', 'Seed Funding', 'Pre-seed Round',
       'Seed Round & Series A', 'Pre Series A', 'Pre seed Round',
       'Angel Round', 'Pre series A1', 'Series E2', 'Seed Round',
       'Bridge Round', 'Pre seed

### ***Insight***
* The sector column contained multivalued cells separated by commas
* Only the first value was retained based on the assumption that they represent the primary sector, while the subsequent values represent additional descriptions
* The variations (like FinTech and Fintech) was also handled

## **Standardizing Funding Stages**

In [27]:
df['stage'] = (
    df['stage']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace('-', ' ', regex=False)
    .str.replace('/', ' ', regex=False)
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)
df['stage'].unique()

array(['seed', 'series a', 'angel', 'series b', 'pre seed',
       'private equity', 'venture series unknown', 'grant',
       'debt financing', 'post ipo debt', 'series h', 'series c',
       'series e', 'corporate round', 'undisclosed',
       'https docsgooglecom spreadsheets d 1x9zineaz6aunchihnmi8u6ks7kntr3byyybgfqaoua editgid1861303593',
       'series d', 'secondary market', 'post ipo equity',
       'non equity assistance', 'funding round', 'nan', 'fresh funding',
       'pre series a', 'series g', 'post series a', 'seed funding',
       'seed fund', 'series f', 'seed round', 'pre series', 'debt',
       'pre series c', 'pre series b', 'bridge', 'series b2', 'edge',
       'seed a', 'series a 1', 'pre seed round', 'seed round series a',
       'angel round', 'pre series a1', 'series e2', 'bridge round',
       'seed investment', 'series d1', 'mid series', 'series c d',
       '1200000', 'series f2', 'series b3', 'pe', 'series f1', '300000',
       'early seed', '6000000', '1000

In [28]:
stage_map = {
    # Seed
    'seed funding':'seed',
    'seed fund':'seed',
    'seed round':'seed',
    'seed investment':'seed',
    'early seed':'seed',
    'seed a':'seed',

    # Angel
    'angel round':'angel',

    # Pre-seed
    'pre seed round':'pre seed',

    # Pre-series A
    'pre series a1':'pre series a',

    # Series variants
    'series a 1':'series a',
    'series a2':'series a',
    'series b2':'series b',
    'series b3':'series b',
    'series c d':'series c',
    'series d1':'series d',
    'series e2':'series e',
    'series f1':'series f',
    'series f2':'series f',

    # Other
    'bridge round':'bridge',
    'debt':'debt financing',
    'pe':'private equity',
    'seies a':'series a',
    'post series a': 'series a',
    'funding round': 'undisclosed',
    'fresh funding': 'undisclosed',
    'seed round series a': 'series a',
    'mid series': 'venture series unknown',
    'edge': np.nan,

    # String nan
    'nan': np.nan
}

df['stage'] = df['stage'].replace(stage_map)

In [29]:
junk = [
    '1200000',
    '300000',
    '1000000',
    '6000000',
    'https docsgooglecom spreadsheets d 1x9zineaz6aunchihnmi8u6ks7kntr3byyybgfqaoua editgid1861303593'
]

df.loc[df['stage'].isin(junk), 'stage'] = np.nan

In [30]:
df['stage'].value_counts()

,count
stage,
seed,695
series a,311
pre series a,291
series b,138
series c,115
pre seed,73
debt financing,58
series d,52
angel,41


### ***Insight***
* The Stage column had a huge number of inconsistencies in formatting, naming convention and spelling differences
* Obvious invalid values like URLs and numeric values were assigned null values

## **Standardizing Funding Amount**

In [31]:
# first clean symbols
df['amount($)'] = (
    df['amount($)']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace(['Undisclosed', '—', 'nan'], np.nan)
)

# numeric conversion
df['amount($)'] = pd.to_numeric(df['amount($)'], errors='coerce')

# convert only 2018 rows from INR to USD
df.loc[df['year'] == 2018, 'amount($)'] /= 68.41
df['amount($)']=df['amount($)'].round()

In [32]:
df.head()

,company_brand,sector,stage,amount($),headquarter,what_it_does,year
0,thecollegefever,Brand Marketing,seed,3654.0,"Bangalore, Karnataka, India","TheCollegeFever is a hub for fun, fiesta and f...",2018
1,happy cow dairy,Agriculture,seed,584710.0,"Mumbai, Maharashtra, India",A startup which aggregates milk from dairy far...,2018
2,myloancare,Credit,series a,950153.0,"Gurgaon, Haryana, India",Leading Online Loans Marketplace in India,2018
3,payme india,Financial Services,angel,29235.0,"Noida, Uttar Pradesh, India",PayMe India is an innovative FinTech organizat...,2018
4,eunimart,E-Commerce Platforms,seed,NaN,"Hyderabad, Andhra Pradesh, India",Eunimart is a one stop solution for merchants ...,2018


### ***Insight***
* The amount column was recorded with different currencies (using INR in 2018 and USD in 2019-2021)
* The currency symbols and the formatting was removed
* The 2018 amounts were converted to USD with an approximate exchange rate of ₹68.41 per USD

## **Standardizing Cities**

In [33]:
df['city']=df['headquarter'].str.strip().str.split(',').str[0]

In [34]:
df['city'].unique()

array(['Bangalore', 'Mumbai', 'Gurgaon', 'Noida', 'Hyderabad',
       'Bengaluru', 'Kalkaji', 'Delhi', 'India', 'Hubli', 'New Delhi',
       'Chennai', 'Mohali', 'Kolkata', 'Pune', 'Jodhpur', 'Kanpur',
       'Ahmedabad', 'Azadpur', 'Haryana', 'Cochin', 'Faridabad', 'Jaipur',
       'Kota', 'Anand', 'Bangalore City', 'Belgaum', 'Thane', 'Margão',
       'Indore', 'Alwar', 'Kannur', 'Trivandrum', 'Ernakulam',
       'Kormangala', 'Uttar Pradesh', 'Andheri', 'Mylapore', 'Ghaziabad',
       'Kochi', 'Powai', 'Guntur', 'Kalpakkam', 'Bhopal', 'Coimbatore',
       'Worli', 'Alleppey', 'Chandigarh', 'Guindy', 'Lucknow', nan,
       'Telangana', 'Gurugram', 'Surat', 'Uttar pradesh', 'Rajasthan',
       'Tirunelveli', 'Singapore', 'Gujarat', 'Kerala', 'Frisco',
       'California', 'Dhingsara', 'New York', 'Patna', 'San Francisco',
       'San Ramon', 'Paris', 'Plano', 'Sydney', 'San Francisco Bay Area',
       'Bangaldesh', 'London', 'Milano', 'Palmwoods', 'France',
       'Samastipur', 'Irvin

In [35]:
df[df['city']=='Online Media\t#REF!']

,company_brand,sector,stage,amount($),headquarter,what_it_does,year,city
2770,sochcast,Sochcast is an Audio experiences company that ...,NaN,NaN,Online Media\t#REF!,"CA Harvinderjit Singh Bhatia, Garima Surana, A...",2021,Online Media\t#REF!


In [36]:
df[df['city']=='Computer Games']
#headquater contains sector values

,company_brand,sector,stage,amount($),headquarter,what_it_does,year,city
1768,fanplay,Computer Games,NaN,NaN,Computer Games,A real money game app specializing in trivia g...,2021,Computer Games
1781,fanplay,Computer Games,NaN,NaN,Computer Games,A real money game app specializing in trivia g...,2021,Computer Games


In [37]:
df[df['city']=='Information Technology & Services']
#swapped sector and city/headquarter values

,company_brand,sector,stage,amount($),headquarter,what_it_does,year,city
2846,peak,Manchester,series c,75000000.0,Information Technology & Services,Peak helps the world's smartest companies put ...,2021,Information Technology & Services


In [38]:
df['city'].unique()

array(['Bangalore', 'Mumbai', 'Gurgaon', 'Noida', 'Hyderabad',
       'Bengaluru', 'Kalkaji', 'Delhi', 'India', 'Hubli', 'New Delhi',
       'Chennai', 'Mohali', 'Kolkata', 'Pune', 'Jodhpur', 'Kanpur',
       'Ahmedabad', 'Azadpur', 'Haryana', 'Cochin', 'Faridabad', 'Jaipur',
       'Kota', 'Anand', 'Bangalore City', 'Belgaum', 'Thane', 'Margão',
       'Indore', 'Alwar', 'Kannur', 'Trivandrum', 'Ernakulam',
       'Kormangala', 'Uttar Pradesh', 'Andheri', 'Mylapore', 'Ghaziabad',
       'Kochi', 'Powai', 'Guntur', 'Kalpakkam', 'Bhopal', 'Coimbatore',
       'Worli', 'Alleppey', 'Chandigarh', 'Guindy', 'Lucknow', nan,
       'Telangana', 'Gurugram', 'Surat', 'Uttar pradesh', 'Rajasthan',
       'Tirunelveli', 'Singapore', 'Gujarat', 'Kerala', 'Frisco',
       'California', 'Dhingsara', 'New York', 'Patna', 'San Francisco',
       'San Ramon', 'Paris', 'Plano', 'Sydney', 'San Francisco Bay Area',
       'Bangaldesh', 'London', 'Milano', 'Palmwoods', 'France',
       'Samastipur', 'Irvin

In [39]:
random=['Computer Games','Food & Beverages','Pharmaceuticals\t#REF!','Gurugram\t#REF!','Online Media\t#REF!','Information Technology & Services']
print('Number of rows with random cities:',df[df['city'].isin(random)].shape[0])
print('Percent of corrupt cities:',(9/2879)*100)

#dropping the random city values
df = df[~df['city'].isin(random)]

Number of rows with random cities: 9
Percent of corrupt cities: 0.3126085446335533


In [40]:
df['city'].unique()

array(['Bangalore', 'Mumbai', 'Gurgaon', 'Noida', 'Hyderabad',
       'Bengaluru', 'Kalkaji', 'Delhi', 'India', 'Hubli', 'New Delhi',
       'Chennai', 'Mohali', 'Kolkata', 'Pune', 'Jodhpur', 'Kanpur',
       'Ahmedabad', 'Azadpur', 'Haryana', 'Cochin', 'Faridabad', 'Jaipur',
       'Kota', 'Anand', 'Bangalore City', 'Belgaum', 'Thane', 'Margão',
       'Indore', 'Alwar', 'Kannur', 'Trivandrum', 'Ernakulam',
       'Kormangala', 'Uttar Pradesh', 'Andheri', 'Mylapore', 'Ghaziabad',
       'Kochi', 'Powai', 'Guntur', 'Kalpakkam', 'Bhopal', 'Coimbatore',
       'Worli', 'Alleppey', 'Chandigarh', 'Guindy', 'Lucknow', nan,
       'Telangana', 'Gurugram', 'Surat', 'Uttar pradesh', 'Rajasthan',
       'Tirunelveli', 'Singapore', 'Gujarat', 'Kerala', 'Frisco',
       'California', 'Dhingsara', 'New York', 'Patna', 'San Francisco',
       'San Ramon', 'Paris', 'Plano', 'Sydney', 'San Francisco Bay Area',
       'Bangaldesh', 'London', 'Milano', 'Palmwoods', 'France',
       'Samastipur', 'Irvin

In [41]:
invalid = [
    'India','Haryana','Kerala','Odisha','Rajasthan','Gujarat',
    'Tamil Nadu','West Bengal','Jharkhand','Bihar','Goa','nan'
]

print('No. of rows with obvious non-cities: ',df[df['city'].isin(invalid)].shape[0])
print((165/2890)*100)
df=df[~df['city'].isin(invalid)]

No. of rows with obvious non-cities:  51
5.709342560553633


In [42]:
df['city'].unique()

array(['Bangalore', 'Mumbai', 'Gurgaon', 'Noida', 'Hyderabad',
       'Bengaluru', 'Kalkaji', 'Delhi', 'Hubli', 'New Delhi', 'Chennai',
       'Mohali', 'Kolkata', 'Pune', 'Jodhpur', 'Kanpur', 'Ahmedabad',
       'Azadpur', 'Cochin', 'Faridabad', 'Jaipur', 'Kota', 'Anand',
       'Bangalore City', 'Belgaum', 'Thane', 'Margão', 'Indore', 'Alwar',
       'Kannur', 'Trivandrum', 'Ernakulam', 'Kormangala', 'Uttar Pradesh',
       'Andheri', 'Mylapore', 'Ghaziabad', 'Kochi', 'Powai', 'Guntur',
       'Kalpakkam', 'Bhopal', 'Coimbatore', 'Worli', 'Alleppey',
       'Chandigarh', 'Guindy', 'Lucknow', nan, 'Telangana', 'Gurugram',
       'Surat', 'Uttar pradesh', 'Tirunelveli', 'Singapore', 'Frisco',
       'California', 'Dhingsara', 'New York', 'Patna', 'San Francisco',
       'San Ramon', 'Paris', 'Plano', 'Sydney', 'San Francisco Bay Area',
       'Bangaldesh', 'London', 'Milano', 'Palmwoods', 'France',
       'Samastipur', 'Irvine', 'Tumkur', 'Newcastle Upon Tyne',
       'Shanghai', 'Jiax

In [43]:
city_stand = {
    'Bangalore':'Bengaluru',
    'Banglore':'Bengaluru',
    'Bangalore City':'Bengaluru',
    'Hyderebad':'Hyderabad',
    'Gurgaon':'Gurugram',
    'Ahmadabad':'Ahmedabad',
    'Rajastan':'Rajasthan',
    'Telugana':'Telangana',
    'Orissia':'Odisha',
    'Cochin':'Kochi',
    'Trivandrum':'Thiruvananthapuram',
    'Samsitpur':'Samastipur',
    'San Franciscao':'San Francisco',
    'New Delhi':'Delhi'
}

df['city'] = df['city'].replace(city_stand)

### ***Insight***
* The headquarter column contained value ranging from just cities to full addresses including city,state and country
* Only the cities were retained in a new column 'city'
* There were a few invalid records and some states and since they accounted only a small percentage of values, they were dropped with negligible difference

In [44]:
df.head()

,company_brand,sector,stage,amount($),headquarter,what_it_does,year,city
0,thecollegefever,Brand Marketing,seed,3654.0,"Bangalore, Karnataka, India","TheCollegeFever is a hub for fun, fiesta and f...",2018,Bengaluru
1,happy cow dairy,Agriculture,seed,584710.0,"Mumbai, Maharashtra, India",A startup which aggregates milk from dairy far...,2018,Mumbai
2,myloancare,Credit,series a,950153.0,"Gurgaon, Haryana, India",Leading Online Loans Marketplace in India,2018,Gurugram
3,payme india,Financial Services,angel,29235.0,"Noida, Uttar Pradesh, India",PayMe India is an innovative FinTech organizat...,2018,Noida
4,eunimart,E-Commerce Platforms,seed,NaN,"Hyderabad, Andhra Pradesh, India",Eunimart is a one stop solution for merchants ...,2018,Hyderabad


## **Data Exploration**

In [45]:
df.shape

(2819, 8)

In [46]:
df.head()

,company_brand,sector,stage,amount($),headquarter,what_it_does,year,city
0,thecollegefever,Brand Marketing,seed,3654.0,"Bangalore, Karnataka, India","TheCollegeFever is a hub for fun, fiesta and f...",2018,Bengaluru
1,happy cow dairy,Agriculture,seed,584710.0,"Mumbai, Maharashtra, India",A startup which aggregates milk from dairy far...,2018,Mumbai
2,myloancare,Credit,series a,950153.0,"Gurgaon, Haryana, India",Leading Online Loans Marketplace in India,2018,Gurugram
3,payme india,Financial Services,angel,29235.0,"Noida, Uttar Pradesh, India",PayMe India is an innovative FinTech organizat...,2018,Noida
4,eunimart,E-Commerce Platforms,seed,NaN,"Hyderabad, Andhra Pradesh, India",Eunimart is a one stop solution for merchants ...,2018,Hyderabad


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2819 entries, 0 to 2878
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   company_brand  2819 non-null   object 
 1   sector         2789 non-null   object 
 2   stage          1895 non-null   object 
 3   amount($)      2277 non-null   float64
 4   headquarter    2705 non-null   object 
 5   what_it_does   2819 non-null   object 
 6   year           2819 non-null   int64  
 7   city           2705 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 198.2+ KB


In [48]:
df.describe()

,amount($),year
count,2.277000e+03,2819.000000
mean,1.203706e+08,2020.028379
std,3.469238e+09,1.085051
min,1.000000e+01,2018.000000
25%,5.430000e+05,2020.000000
50%,2.411928e+06,2020.000000
75%,1.000000e+07,2021.000000
max,1.500000e+11,2021.000000


In [49]:
df[df['sector'].isnull()].head()

,company_brand,sector,stage,amount($),headquarter,what_it_does,year,city
58,missmalini entertainment,NaN,seed,1520246.0,"Mumbai, Maharashtra, India",MissMalini Entertainment is a multi-platform n...,2018,Mumbai
105,jagaran microfin,NaN,debt financing,8039760.0,"Kolkata, West Bengal, India",Jagaran Microfin is a Microfinance institution...,2018,Kolkata
121,fleeca,NaN,seed,NaN,"Jaipur, Rajasthan, India",FLEECA is a Tyre Care Provider company.,2018,Jaipur
146,wheelsemi,NaN,series b,204648.0,"Pune, Maharashtra, India","WheelsEMI is the brand name of NBFC, WheelsEMI...",2018,Pune
153,fric bergen,NaN,venture series unknown,NaN,"Alwar, Rajasthan, India",Fric Bergen is a leader in the specialty food ...,2018,Alwar


In [50]:
df.isnull().sum()

,0
company_brand,0
sector,30
stage,924
amount($),542
headquarter,114
what_it_does,0
year,0
city,114


### *Handling missing values*
* Even though the sector column has 30 missing values, the 'what_it_does' column still provides the useful context information
*  Missing values in the 'stage' and 'amount($)' columns may represent unavailable information rather than data entry errors and hence decided against imputing them



In [51]:
df.duplicated().sum()

np.int64(23)

In [52]:
df[df.duplicated(keep=False)].sort_values('company_brand').head()

,company_brand,sector,stage,amount($),headquarter,what_it_does,year,city
1787,advantage club,HRTech,NaN,1700000.0,Mumbai,Advantage Club is India's largest employee eng...,2021,Mumbai
1774,advantage club,HRTech,NaN,1700000.0,Mumbai,Advantage Club is India's largest employee eng...,2021,Mumbai
1922,asqi advisors,Financial Services,pre series a,1000000.0,Mumbai,Bringing Blockchain technology intro mainstrea...,2021,Mumbai
1908,asqi advisors,Financial Services,pre series a,1000000.0,Mumbai,Bringing Blockchain technology intro mainstrea...,2021,Mumbai
1779,bewakoof,Apparel & Fashion,NaN,8000000.0,Mumbai,Bewakoof is a lifestyle fashion brand that mak...,2021,Mumbai


In [53]:
#dropping duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)

In [54]:
df = df[['company_brand','sector','stage','amount($)','what_it_does','year','city']]

In [55]:
df.rename(columns={'amount($)':'amount_usd'}, inplace=True)

## **Final Cleaned Dataset Summary**
* Total records: 2796
* Features: 8
* Time period: 2018-2021
*Missing values were retained where absence represented genuinely unavailable information rather than data quality issues
* The final dataset was standardized to ensure consistency across years and facilitate further analysis

In [56]:
#saving the cleaned dataset for analysis using MySQL
df.to_csv('startup_cleaned.csv',index=False)

In [57]:
from google.colab import files
#files.download('startup_cleaned.csv')

In [58]:
df

,company_brand,sector,stage,amount_usd,what_it_does,year,city
0,thecollegefever,Brand Marketing,seed,3654.0,"TheCollegeFever is a hub for fun, fiesta and f...",2018,Bengaluru
1,happy cow dairy,Agriculture,seed,584710.0,A startup which aggregates milk from dairy far...,2018,Mumbai
2,myloancare,Credit,series a,950153.0,Leading Online Loans Marketplace in India,2018,Gurugram
3,payme india,Financial Services,angel,29235.0,PayMe India is an innovative FinTech organizat...,2018,Noida
4,eunimart,E-Commerce Platforms,seed,NaN,Eunimart is a one stop solution for merchants ...,2018,Hyderabad
...,...,...,...,...,...,...,...
2791,gigforce,Staffing & Recruiting,pre series a,3000000.0,A gig/on-demand staffing company.,2021,Gurugram
2792,vahdam,Food & Beverages,series d,20000000.0,VAHDAM is among the world’s first vertically i...,2021,Delhi
2793,leap finance,Financial Services,series c,55000000.0,International education loans for high potenti...,2021,Bengaluru
2794,collegedekho,Edtech,series b,26000000.0,"Collegedekho.com is Student’s Partner, Friend ...",2021,Gurugram
